# Quality Control (GenoTools)


Pipeline: https://github.com/dvitale199/GenoTools/tree/main/genotools

**Version**: 1.0.0  


**Last iteration**: 22-MAR-2026  

## Imports


In [ ]:
import os
import subprocess
from datetime import date, datetime
d = date.today()

## Set directories and variables

### Common paths

In [ ]:
# Directories

# Home
home = "/path/to/home"

# Hestia NGS Software
tools = "/path/to/tools"

# Main directory
main_dir = "/path/to/home/analysis"
MAIN_DIR = main_dir  ### alias

# Data directory
data_dir = f"{main_dir}/output/data"
DATA_DIR = data_dir  ### alias

# Raw data directory
raw_dir = f"{data_dir}/RAW"
RAW_DIR = raw_dir  ### alias

# Imputed data directory
impt_dir = f"{data_dir}/IMPUTED"
IMPT_DIR = raw_dir  ### alias

# Meta data (covariate, population, ancestry labels, etc.)
meta_dir = f"{data_dir}/META"
META_DIR = meta_dir  ### alias

### Paths to software and tools

In [ ]:
# Plink1.9 and Plink2.0 path
plink1 = f"{tools}/plink_linux_x86_64_20250615/plink"
plink2 = f"{tools}/plink2_linux_avx2_20250609/plink2"

### Input, output, covariate files

In [ ]:
# Input file path without suffix
rawFile = f"{raw_dir}/CATPD_master_key.update_pheno"
inputPfile = f"{raw_dir}/CATPD_master_key"  ### Updated IDs, Sex and added PD Pheno

# Covariate file
covar = f"{meta_dir}/CATPD_master_key.cov"

# Populations/Ancestries file
pops = f"{meta_dir}/CATPD_master_key.pop"

# Pheno Name
pheno = "STATUS"

# Keep file / added CAS
keep = f"{meta_dir}/CATPD_master_key.keep"

# Chromosomes as list
chromosomes = list(range(1, 23))

# Output directory path and output prefix
outdir = f"{raw_dir}/QC_GenoTools"
os.makedirs(outdir, exist_ok=True)
output = "CATPD"

### SNPs to exclude defined by GP2 and LARGE-PD

In [ ]:
# SNPs to exclude
# GP2 underperforming
snps_gp2_underperf = f"{meta_dir}/underperforming_snps.txt"

### Reference genomes and other reference files

## Exclude GP2 underperforming SNPs and LARGE-PD SNPs

In [ ]:
def excludeSNPs(rawFile, inputPfile, snps_gp2_underperf):
    snps_GP2 = [
        "plink2",
        "--pfile",
        f"{rawFile}.snps.largepd",
        "--exclude",
        snps_gp2_underperf,
        "--make-pgen",
        "--out",
        f"{outdir}/{output}",
    ]
    subprocess.run(snps_GP2, check=True)

In [ ]:
excludeSNPs(rawFile, inputPfile, snps_gp2_underperf)

## GenoTools

### Define paths

In [ ]:
# Home dir
home = "/path/to/home"

# Reference files directory
ref_files = f"{home}/CAT-PD/TOOLS/MODEL"

# Reference panel and labels
ref_panel = f"{ref_files}/ref_panel_gp2_prune_rm_underperform_pos_update"
ref_labels = f"{ref_files}/ref_panel_ancestry_updated.txt"

# Pretrained model (if supervised)
model_path = f"{ref_files}/CATPD_ancestry_umap_linearsvc_ancestry_model.pkl"

# Input
output = "CATPD_before_qc"
input_path = f"{outdir}/{output}"

# Output
outdir = f"{home}/GenoTools_QC"

# Input and output files
input_path = f"{outdir}/{output}"
output_path = f"{outdir}/{output}"

In [ ]:
def run_genotools(
    input_path,
    output_path,
    ref_panel,
    ref_labels,
    model_path=None,
    model_type="supervised",
    runtype="print",
    genotools="/path/to/home/miniconda3/envs/genotools/bin/genotools",
):

    # QC and ancestry prediction with pretrained model
    genotools_supervised = [
        genotools,
        "--pfile", input_path,
        "--out", output_path,
        "--ancestry", "True",
        "--model", model_path,
        "--ref_panel", ref_panel,
        "--ref_labels", ref_labels,
        "--full_output", "True",
        "--all_sample", "True",
        "--all_variant", "True",
    ]

    # QC and train new model for ancestry prediction
    genotools_train_model = [
        genotools,
        "--pfile", input_path,
        "--out", output_path,
        "--ancestry", "True",
        "--ref_panel", ref_panel,
        "--ref_labels", ref_labels,
        "--full_output", "True",
        "--all_sample", "True",
        "--all_variant", "True",
    ]

    # Get timestamp
    ts = datetime.now().strftime("%m%d%Y_%H_%M_%S")

    # Select either print the command for bash or run in notebook
    def run_model(model_type, runtype="print"):
        if runtype == "print":
            print(
                f"nohup {' '.join(model_type)} > {output_path}_{ts}.out 2> {output_path}_{ts}.err &"
            )
        elif runtype == None:
            subprocess.run(model_type, check=True)

    if model_type == "supervised":
        run_model(genotools_supervised)
    elif model_type == "unsupervised":
        run_model(genotools_train_model)
    else:
        raise Exception(
            "Invalid model selected.\nChoose supervised with pretrained model or unsupervised."
        )

<div class="alert alert-block alert-info">
<b>Tip:</b> Make sure the phenotype in psam file is named "PHENO1".
</div>

In [ ]:
run_genotools(
    input_path,
    output_path,
    ref_panel,
    ref_labels,
    model_path=None,
    model_type="unsupervised",
)